# Preprocessing Data

**Importing libraries and reading the dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Once your Drive is mounted, you can access files using the path `/content/drive/My Drive/`. Replace `your_folder/your_file.csv` with the actual path to your file within Google Drive. For example, if your file is in a folder named 'my_data' in your Drive, the path would be `/content/drive/My Drive/my_data/your_file.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


# Example: Reading a CSV file from Google Drive
# Replace 'your_folder/your_file.csv' with the actual path to your file
file_path = '/content/drive/My Drive/playground-series-s6e1/train.csv'
file_path2 = '/content/drive/My Drive/playground-series-s6e1/test.csv'
try:
    df = pd.read_csv(file_path)
    print("File loaded successfully!")
    dft = pd.read_csv(file_path2)
    print("File loaded successfully!")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the path is correct and your Drive is mounted.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
df.head()

In [ ]:
dft.head()

**Check for the data sanity**

(Any missing values? duplicates? garbage values?)

In [ ]:
df.shape

In [ ]:
dft.shape # it should be 12 columns because we excluded the exam score

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
dft.isnull().sum()

No missing values

In [ ]:
df.duplicated().sum()

In [ ]:
dft.duplicated().sum()

No duplicates

In [ ]:
for i in df.select_dtypes(include=['object']).columns:
    print(df[i].value_counts())
    print("***" * 10)

No garbage values

**EDA**

In [ ]:
df.describe() #for numerical features

In [ ]:
df.describe(include='object') #for categorical features

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# To understand the data distribution
for i in df.select_dtypes(include="number").columns:
  sns.histplot(data= df, x= i)
  plt.show()

In [ ]:
# To find any outliers
for i in df.select_dtypes(include="number").columns:
  sns.boxplot(data= df, x= i)
  plt.show()

No outliers

In [ ]:
for i in ['age', 'study_hours', 'class_attendance', 'sleep_hours']:

# Diff ways of plotting since there's an overplotting

  #sns.scatterplot(data= df, x= i, y= 'exam_score', alpha=0.01, s=1)
  #plt.show()

  sample = df.sample(n=5000)  # Plot only 5000 random points
  plt.scatter(sample[i], sample['exam_score'], alpha=0.3)
  plt.show()

  #plt.hexbin(data= df, x= i, y= 'exam_score', gridsize=30, cmap='Blues')
  #plt.colorbar(label='count')
  #plt.show()

  #plt.hist2d(data= df, x= i, y= 'exam_score', bins=50, cmap='Blues')
  #plt.colorbar()
  #plt.show()

**Notes**

**Age:**
a very weak or no correlation between the two variables,
There's no clear diagonal pattern that would indicate a strong relationship. The data forms vertical stripes, meaning for each x-value (17, 18, 19, etc.), exam scores are spread across the full range from ~20 to 100. This suggests the x-variable doesn't predict exam performance well. And the **AGE**'s got a **uniform distribution**. Plus, The **Age** appears to be **discrete** integers (whole numbers), which is why you see distinct vertical columns rather than a continuous spread.

**Study hours:**
a strong positive correlation between study hours and exam scores. As study hours increase, exam scores generally increase,  the plot gets very dense in the upper right (7-8 study hours, 90-100 exam scores). Students who study more tend to score higher.

**Class attendance:** a moderate to strong positive correlation between class attendance and exam scores. In fact, high attendance doesn't guarantee high scores - Plenty of students with 90%+ attendance still score in the 60s, 70s, or 80s. It's not a tight relationship.

**Sleep hours:** almost no correlation, Whether students sleep 4 hours or 10 hours, exam scores are distributed fairly evenly from about 20 to 100. Sleep duration doesn't seem to predict performance.

**Encoding**

**One-hot encoding:** for features that have no natural order

In [ ]:
dfe = pd.get_dummies(data= df, columns=['gender', 'course', 'internet_access', 'study_method'], dtype= int)
dfte = pd.get_dummies(data= dft, columns=['gender', 'course', 'internet_access', 'study_method'], dtype= int)
dfte = dfte.reindex(columns=dfe.columns, fill_value=0)

In [ ]:
dfe

**Label encoding:** for ranked / natural ordered features

In [ ]:
from sklearn.preprocessing import LabelEncoder
columns_to_encode = ['sleep_quality', 'facility_rating', 'exam_difficulty']

for c in columns_to_encode:
  le = LabelEncoder()
  dfe[c] = le.fit_transform(df[c])
  dfte[c] = le.transform(dft[c])
  dfe = dfe.drop(c, axis=1)
  dfte = dfte.drop(c, axis=1)


In [ ]:
dfe

In [ ]:
dfte

In [ ]:
dfte = dfte.drop('exam_score', axis=1)

**Max-Min Scaling**

In [ ]:
column_names = dfe.columns.tolist()
index_values = dfe.index.tolist()
column_names1 = dfte.columns.tolist()
index_values1 = dfte.index.tolist()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
s1 = MinMaxScaler()
s2 = MinMaxScaler()

dfe = s1.fit_transform(dfe)
dfte = s2.fit_transform(dfte)

In [ ]:
dfes = pd.DataFrame(dfe, columns=column_names, index=index_values)
dftes = pd.DataFrame(dfte, columns=column_names1, index=index_values1)

In [ ]:
dfes

In [ ]:
dftes

In [ ]:
# Check the correlation
plt.figure(figsize=(20, 18))
sns.heatmap(dfes.corr(), cmap='coolwarm', annot=True)
plt.show()

# Building the Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score

X = dfes.drop('exam_score', axis=1)
y = dfes['exam_score']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [183]:
base_models = [
    ("lr", LinearRegression()),

    ("svm", SVR(
        kernel="rbf",
        C=10,
        epsilon=0.1
    )),

    ("rf", RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )),

    ("xgb", XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
]

meta_model = LinearRegression()

stack = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=1
)

stack.fit(X_train, y_train)

y_pred = stack.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Stacking RMSE:", rmse)

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGKILL(-9)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

**Linear Regression**

In [ ]:
X = dfe.drop('exam_score', axis=1)
y = dfe['exam_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42),
    'XGBoost': XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        tree_method='hist',
        n_jobs=-1,
        random_state=42
    ),
    'SVM': SVR(kernel='rbf')
}

for name, model in models.items():
    model.fit(X_train, y_train)

    # Predict (still scaled)
    y_pred_scaled = model.predict(X_test)

    # 🔑 Inverse scale to original exam score units
    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
    y_true = y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()

    # MSE in original units
    mse = mean_squared_error(y_true, y_pred)

    print(f"{name}: Mean Squared Error = {mse:.4f}")


In [ ]:
from sklearn.model_selection import train_test_split, root_mean_squared_error
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        n_jobs=-1,
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        tree_method="hist",   # very important for speed
        n_jobs=-1,
        random_state=42
    )
}

kf = KFold(n_splits=3, shuffle=True, random_state=42)

y_mean = np.mean(y)  # for percentage interpretation (optional)

for name, model in models.items():
    rmse_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        rmse = np.root_mean_squared_error(y_val, y_pred)
        rmse_scores.append(rmse)

    rmse_scores = np.array(rmse_scores)

    rmse_mean = rmse_scores.mean()
    rmse_std = rmse_scores.std()
    rmse_percent = (rmse_mean / y_mean) * 100

    print(f"{name}")
    print(f"  RMSE: {rmse_scores}")